In [1]:
import requests

In [2]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)


In [3]:
len(zip_response.content)

17545754

In [4]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [5]:
zip_archive

<zipfile.ZipFile file=<_io.BytesIO object at 0x10822e570> mode='r'>

In [ ]:
filenames = zip_archive.namelist()
filenames[20:30]

In [7]:
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')

In [ ]:
print(mdx_content)

In [8]:
import frontmatter
post = frontmatter.loads(mdx_content)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [ ]:
post.metadata

In [ ]:
post.content

In [10]:
_, filename_corrected = filename.split('/', maxsplit=1)
print(filename_corrected)

docs/platform/alerts.mdx


In [11]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [12]:
documents = []
with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zip_ref:
    for file_path in zip_ref.namelist():
        if not file_path.endswith(('.md', '.mdx')):
            continue
        with zip_ref.open(file_path) as file:
            content = file.read().decode('utf-8')
            post = frontmatter.loads(content)
            doc = {
                'content': post.content,
                'title': post.metadata.get('title'),
                'description': post.metadata.get('description'),
                'filename': filename_corrected
            }
            documents.append(doc)                                   

In [ ]:
len(documents)

In [13]:
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()
    documents = []
    with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc) 
    return documents

In [14]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [15]:
document = documents[10]
print(document)

{'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python\nmy_report.save_html(“file.html”)\n```\n\nThis option is useful for sharing Reports 

In [16]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")


Loaded 95 documents


In [17]:
document = files[10]

print(document.filename)
print(document.content[:160])

docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:




In [18]:
query = 'LLM as a Judge'

In [ ]:
!uv add minsearch

In [19]:
from minsearch import Index

index = Index(
    text_fields = ["title", "description", "content"],
    keyword_fields = ["filename"]
)
index.fit(documents)

In [ ]:
results = index.search(query, num_results=5)

In [ ]:
len(results)

In [ ]:
len(results[0]['content']) #20K characters

rag:

1. search <-- 5docs
2. prompt <00 5 * 20k = 100k
3. llm

In [ ]:
document = list(range(0, 100))

In [ ]:
window_size = 10
start = 0
while start < len(document):
    end = start + window_size
    chunk = document[start:end]
    print(chunk)
    start = end

In [ ]:
doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

In [ ]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})
        
        start = end - step
        
        if end >= text_length:
            break
    return chunks

In [ ]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

In [ ]:
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')
    chunks = sliding_window(content, size=3000, step=1500)
    
    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [ ]:
document_chunks[10]

In [ ]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [ ]:
result = chunk_index.search(query)

In [ ]:
result

In [ ]:
from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=3000, step=1500)

## RAG

In [ ]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
search_result = chunk_index.search(query, num_results=5)

In [ ]:
query = 'how do I implement llm as a judge?'

In [ ]:
import json

search_result_json = json.dumps(search_result, indent=2)

In [ ]:
instructions = """
You're a course assistant, 
your task is to answer the QUESTION 
from the course students using the provided CONTEXT
"""
user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [ ]:
print(user_prompt)

In [ ]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    
    messages = []
    if instructions is not None:
        messages.append({"role": "system", "content": instructions})
        
    messages.append({"role": "user", "content": user_prompt})
    
    
    response = openai_client.responses.create(
        model=model,
        input=messages
    )
    return response.output_text

In [ ]:
answer = llm(user_prompt, instructions)

In [ ]:
print(answer)

In [ ]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [ ]:
instructions = """
You're a course assistant, 
your task is to answer the QUESTION 
from the course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)
    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()
    return user_prompt

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [ ]:
rag('how do I implement llm as a judge')